<a href="https://colab.research.google.com/github/thinus283-ux/LR/blob/main/Mcmc_Hubble_s8_tention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# =====================================================
# Logic Relativity - FULL ADVANCED TEST
# CLASS + MCMC with Real Data (H0 + S8)
# =====================================================

# === 1. Install all required packages ===
!pip install classy emcee tqdm --quiet

import numpy as np
import emcee
from classy import Class
from tqdm import tqdm
import matplotlib.pyplot as plt

print("=== Logic Relativity + CLASS + MCMC (Full Advanced) ===\n")

# === Real Data ===
H0_planck = 67.4
H0_sh0es = 73.04
sigma_H0_planck = 0.5
sigma_H0_sh0es = 1.04

S8_planck = 0.834
S8_kids = 0.759
sigma_S8_planck = 0.016
sigma_S8_kids = 0.024

def log_likelihood(theta):
    K_s, thinning, env_factor, Omega_m, H0 = theta

    # Safety checks
    if not (0.22 < Omega_m < 0.42 and 62 < H0 < 78):
        return -np.inf

    cosmo = Class()
    try:
        cosmo.set({
            'Omega_cdm': Omega_m - 0.049,
            'Omega_b': 0.049,
            'h': H0 / 100.0,
            'n_s': 0.965,
            'A_s': 2.1e-9,
            'output': 'mPk',
            'P_k_max_h/Mpc': 1.0,
            'z_max_pk': 3.0
        })
        cosmo.compute()

        sigma8 = cosmo.sigma8()
        cosmo.struct_cleanup()
        cosmo.empty()

    except:
        return -np.inf

    # Logic Relativity effects
    growth_suppression = 1 - env_factor * 0.065
    S8_theory = sigma8 * np.sqrt(Omega_m / 0.3) * growth_suppression
    H0_theory = H0 * (1 + thinning * K_s)

    # Chi-squared
    chi2_H0 = ((H0_theory - H0_sh0es)**2 / sigma_H0_sh0es**2 +
               (H0 - H0_planck)**2 / sigma_H0_planck**2)

    chi2_S8 = ((S8_theory - S8_kids)**2 / sigma_S8_kids**2 +
               (S8_theory - S8_planck)**2 / sigma_S8_planck**2)

    return -0.5 * (chi2_H0 + chi2_S8)

def log_prior(theta):
    K_s, thinning, env_factor, Omega_m, H0 = theta
    if (0.55 < K_s < 1.25 and
        0.0 < thinning < 0.22 and
        0.72 < env_factor < 0.97 and
        0.26 < Omega_m < 0.36 and
        64 < H0 < 76):
        return 0.0
    return -np.inf

def log_probability(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta)

# === MCMC Setup ===
ndim = 5
nwalkers = 28
nsteps = 650

pos = np.array([0.82, 0.105, 0.90, 0.315, 70.0]) + 0.008 * np.random.randn(nwalkers, ndim)

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability)

print("Running MCMC (this will take time)...")
sampler.run_mcmc(pos, nsteps, progress=True)

# === Results ===
flat_samples = sampler.get_chain(discard=160, thin=8, flat=True)
best_fit = np.median(flat_samples, axis=0)

print("\n=== Best Fit Parameters ===")
print(f"K_s        : {best_fit[0]:.3f}")
print(f"thinning   : {best_fit[1]:.3f}")
print(f"env_factor : {best_fit[2]:.3f}")
print(f"Omega_m    : {best_fit[3]:.3f}")
print(f"H0         : {best_fit[4]:.2f}")

# Final tensions
K_s, thinning, env_factor, Omega_m, H0 = best_fit
H0_theory = H0 * (1 + thinning * K_s)
growth_suppression = 1 - env_factor * 0.065
S8_theory = S8_planck * growth_suppression * np.sqrt(Omega_m / 0.3)

tension_h0 = abs(H0_theory - H0_sh0es) / np.sqrt(sigma_H0_planck**2 + sigma_H0_sh0es**2)
tension_s8 = abs(S8_theory - S8_kids) / np.sqrt(sigma_S8_planck**2 + sigma_S8_kids**2)

print("\n=== Final Tensions ===")
print(f"Hubble tension : {tension_h0:.2f} σ")
print(f"S8 tension     : {tension_s8:.2f} σ")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 17.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 2.9 MB/s eta 0:00:00
=== Logic Relativity + CLASS + MCMC (Full Advanced) ===

Running MCMC (this will take time)...


100%|██████████| 650/650 [1:19:56<00:00,  7.38s/it]


=== Best Fit Parameters ===
K_s        : 0.854
thinning   : 0.098
env_factor : 0.849
Omega_m    : 0.319
H0         : 67.52

=== Final Tensions ===
Hubble tension : 0.14 σ
S8 tension     : 1.84 σ
